# Phase 5 & 6 — Data Validation & Exploratory Data Analysis (EDA)

We load PneumoniaMNIST **directly into memory**, create our strict 100-image educational subset, then explore the data visually and statistically before training.

In [ ]:
# ── Step 0: Auto-install packages ────────────────────────────────────────────
import subprocess, sys
for pkg in ['numpy', 'matplotlib', 'seaborn', 'medmnist', 'Pillow']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('✅ Packages ready.')

In [ ]:
# ── Step 1: Imports ───────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import medmnist
from medmnist import INFO

SEED = 42
SAMPLES_PER_CLASS = 50
DATA_CACHE_DIR = '../data/'

In [ ]:
# ── Step 2: Load Full Dataset Into Memory ─────────────────────────────────────
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])

print('Loading PneumoniaMNIST into memory (cached after first run)...')
dataset = DataClass(split='train', download=True, root=DATA_CACHE_DIR)

all_images = dataset.imgs
all_labels = dataset.labels.squeeze()

print(f'Loaded {len(all_images)} total training images into RAM.')
print(f'Image shape: {all_images[0].shape}')

In [ ]:
# ── Step 3: Create Strict 100-Image Reproducible Subset ───────────────────────
np.random.seed(SEED)
subset_indices = []

for class_id in [0, 1]:
    class_idx = np.where(all_labels == class_id)[0]
    chosen = np.random.choice(class_idx, SAMPLES_PER_CLASS, replace=False)
    subset_indices.extend(chosen)

X_tiny = all_images[subset_indices]
y_tiny = all_labels[subset_indices]

print(f'Subset: {X_tiny.shape} images | Labels: {dict(zip(*np.unique(y_tiny, return_counts=True)))}')

In [ ]:
# ── Step 4: Validate Class Balance ────────────────────────────────────────────
assert len(X_tiny) == 100, f'Expected 100 images, got {len(X_tiny)}'
assert np.sum(y_tiny == 0) == 50, 'Expected exactly 50 Normal samples'
assert np.sum(y_tiny == 1) == 50, 'Expected exactly 50 Pneumonia samples'

print('✅ Validation passed: 50 Normal / 50 Pneumonia, perfectly balanced.')

In [ ]:
# ── Step 5: Visualize Sample Images ───────────────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle('PneumoniaMNIST Subset — 28×28 Grayscale X-Rays', fontsize=13, fontweight='bold')

label_names = {0: 'Normal', 1: 'Pneumonia'}
colors = {0: '#4CAF50', 1: '#F44336'}

for row, cls in enumerate([0, 1]):
    cls_indices = np.where(y_tiny == cls)[0][:6]
    for col, idx in enumerate(cls_indices):
        ax = axes[row, col]
        ax.imshow(X_tiny[idx], cmap='gray')
        ax.axis('off')
        if col == 0:
            ax.set_title(f'{label_names[cls]} ({cls})', fontsize=10,
                         fontweight='bold', color=colors[cls])

plt.tight_layout()
plt.show()

In [ ]:
# ── Step 6: Pixel Statistics ───────────────────────────────────────────────────
print('=== Pixel Statistics (Raw 0-255) ===')
print(f'Min   : {np.min(X_tiny)}')
print(f'Max   : {np.max(X_tiny)}')
print(f'Mean  : {np.mean(X_tiny):.2f}')
print(f'Std   : {np.std(X_tiny):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(X_tiny.flatten(), bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Overall Pixel Distribution')
axes[0].set_xlabel('Pixel Intensity (0–255)')
axes[0].set_ylabel('Frequency')

# Per-class pixel mean comparison
mean_normal = np.mean(X_tiny[y_tiny == 0])
mean_pneumonia = np.mean(X_tiny[y_tiny == 1])
axes[1].bar(['Normal (0)', 'Pneumonia (1)'], [mean_normal, mean_pneumonia],
            color=['#4CAF50', '#F44336'])
axes[1].set_title('Mean Pixel Intensity per Class')
axes[1].set_ylabel('Mean Pixel Value')
for i, v in enumerate([mean_normal, mean_pneumonia]):
    axes[1].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✅ EDA Complete! → Next: Open 02_train_baseline_and_cnn.ipynb')